# DuckDB 101 — From SQL to Python

We just ran the full SQL tour (reading files, aggregations, joins, exports) live in DuckDB's built-in UI. This notebook **picks up from there** — same three tables, same dataset — and adds the two things only Python gives you: the two-way pandas bridge, and real charts.

**Dataset:** US Airline On-Time Performance (BTS) — 2022, 2023, 2024 · ~21 million flights · tables: `flights`, `airlines`, `airports`

## Setup

Reconnect in Python and reload the same three tables — a few lines, not a teaching moment.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import time, os
from pathlib import Path

DATA = Path('../data/flights')

# Persistent DB file, not in-memory — run this cell once ahead of time to "pre-bake" it.
# After the first run, reconnecting here is instant (no CSV re-parse) even after a kernel restart.
con = duckdb.connect(str(DATA / 'flights.duckdb'))

existing = con.sql("SELECT table_name FROM duckdb_tables()").df()['table_name'].tolist()
if 'flights' not in existing:
    print('First run — loading ~21M rows from CSV, this takes about a minute...')
    con.sql(f"CREATE OR REPLACE TABLE airlines AS SELECT * FROM '{DATA}/airlines.csv'")
    con.sql(f"CREATE OR REPLACE TABLE airports AS SELECT * FROM '{DATA}/airports.csv'")
    con.sql(f"CREATE OR REPLACE TABLE flights  AS SELECT * FROM '{DATA}/flights_*.csv'")
else:
    print('Tables already loaded from a previous run — instant.')

con.sql("SELECT table_name, format_bytes(estimated_size) AS estimated_size FROM duckdb_tables()").df()

## 1. From DuckDB to pandas — and back

This is the part that matters for a notebook workflow: DuckDB isn't a separate silo from Python. `.df()` doesn't just print a table — it hands back a **real pandas DataFrame** that lives in memory and that the rest of your notebook can keep using.

Below, `airline_summary` is the DuckDB query result. We assign it to a variable and then do ordinary pandas work on it — no SQL involved.

In [ ]:
airline_summary = con.sql("""
    SELECT
        al.AIRLINE,
        COUNT(*)                                          AS total_flights,
        ROUND(AVG(f.ArrDelay), 1)                         AS avg_arr_delay_min,
        ROUND(SUM(f.Cancelled) * 100.0 / COUNT(*), 1)    AS cancel_pct
    FROM flights f
    JOIN airlines al ON f.Reporting_Airline = al.IATA_CODE
    GROUP BY al.AIRLINE
    ORDER BY avg_arr_delay_min DESC
""").df()

type(airline_summary)

In [ ]:
# Plain pandas from here on — no SQL
airline_summary.describe()

In [ ]:
# A quick pandas-native chart straight off the DuckDB result
airline_summary.sort_values("avg_arr_delay_min").plot(
    kind="barh", x="AIRLINE", y="avg_arr_delay_min",
    legend=False, figsize=(7, 6), color="steelblue",
    title="Average Arrival Delay by Airline (2022–2024)",
)

## 2. Query a pandas DataFrame with SQL

The bridge runs both ways. DuckDB can run SQL **directly against any DataFrame already in memory** — just reference the Python variable name as if it were a table. No export, no re-import.

Here we build a small lookup table by hand in pandas (`delay_severity`), then join it against `airline_summary` — a DuckDB result — purely with SQL.

In [ ]:
delay_severity = pd.DataFrame({
    "tier":      ["On-time", "Minor", "Moderate", "Severe"],
    "min_delay": [-100,      0,       10,         20],
    "max_delay": [0,         10,      20,         1000],
})

delay_severity

In [ ]:
# SQL querying two in-memory DataFrames at once — airline_summary and delay_severity
con.sql("""
    SELECT
        a.AIRLINE,
        a.avg_arr_delay_min,
        s.tier AS delay_tier
    FROM airline_summary a
    JOIN delay_severity s
      ON a.avg_arr_delay_min >= s.min_delay
     AND a.avg_arr_delay_min <  s.max_delay
    ORDER BY a.avg_arr_delay_min DESC
""").df()

## 3. Now Let's Visualize It

We already saw these numbers as tables in the UI. Python's the tool for turning them into charts.

In [ ]:
# Monthly delay trend across all 3 years
monthly = con.sql("""
    SELECT
        strftime(FlightDate::DATE, '%Y-%m') AS month,
        COUNT(*)                             AS flights,
        ROUND(AVG(ArrDelay), 1)              AS avg_arr_delay
    FROM flights
    WHERE Cancelled = 0
    GROUP BY month
    ORDER BY month
""").df()

monthly.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly['month'], monthly['avg_arr_delay'], color='steelblue', width=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Arrival Delay by Month (2022–2024)', fontsize=13)
ax.set_ylabel('Minutes')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

## 4. What's Actually Causing Delays?

In [ ]:
con.sql("""
    SELECT
        al.AIRLINE,
        ROUND(AVG(f.CarrierDelay), 1)      AS carrier,
        ROUND(AVG(f.WeatherDelay), 1)      AS weather,
        ROUND(AVG(f.NASDelay), 1)          AS air_system,
        ROUND(AVG(f.LateAircraftDelay), 1) AS late_aircraft,
        ROUND(AVG(f.SecurityDelay), 1)     AS security
    FROM flights f
    JOIN airlines al ON f.Reporting_Airline = al.IATA_CODE
    WHERE f.ArrDelay > 0 AND f.Cancelled = 0
    GROUP BY al.AIRLINE
    ORDER BY carrier DESC
""").df()

## 5. The Southwest December 2022 Story

Southwest cancelled ~16,700 flights in one week. Let's find it.

In [ ]:
wn = con.sql("""
    SELECT
        al.AIRLINE,
        strftime(FlightDate::DATE, '%Y-%m') AS month,
        COUNT(*)                             AS flights,
        SUM(Cancelled)                       AS cancellations,
        ROUND(SUM(Cancelled) * 100.0 / COUNT(*), 1) AS cancel_pct
    FROM flights f
    JOIN airlines al ON f.Reporting_Airline = al.IATA_CODE
    WHERE al.AIRLINE = 'Southwest Airlines'
    GROUP BY al.AIRLINE, month
    ORDER BY month
""").df()

wn

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(wn['month'], wn['cancel_pct'], color='steelblue', width=0.7)

# Highlight December 2022
for i, m in enumerate(wn['month']):
    if m == '2022-12':
        bars[i].set_color('firebrick')
        ax.annotate('Dec 2022\nSouthwest meltdown',
                    xy=(i, wn['cancel_pct'].iloc[i]),
                    xytext=(i + 1.5, wn['cancel_pct'].iloc[i]),
                    fontsize=9, color='firebrick',
                    arrowprops=dict(arrowstyle='->', color='firebrick'))

ax.set_title('Southwest Airlines — Cancellation Rate by Month', fontsize=13)
ax.set_ylabel('Cancellation %')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

## 6. Worst Airports to Depart From

In [ ]:
con.sql("""
    SELECT
        ap.iata_code,
        ap.name,
        ap.municipality || ', ' || ap.iso_region AS location,
        COUNT(*)                        AS departures,
        ROUND(AVG(f.DepDelay), 1)       AS avg_dep_delay,
        ROUND(SUM(f.Cancelled) * 100.0 / COUNT(*), 1) AS cancel_pct
    FROM flights f
    JOIN airports ap ON f.Origin = ap.iata_code
    GROUP BY ap.iata_code, ap.name, location
    HAVING departures > 10000
    ORDER BY avg_dep_delay DESC
    LIMIT 15
""").df()

## 7. Local Spotlight: Albany & LaGuardia

Zooming into Albany and LaGuardia for this audience — same tables, same SQL, just a `WHERE` clause away.

In [ ]:
# Overview: volume, delays, cancellations for ALB and LGA
con.sql("""
    SELECT
        ap.iata_code,
        ap.name,
        COUNT(*)                                       AS total_flights,
        ROUND(AVG(f.DepDelay), 1)                      AS avg_dep_delay,
        ROUND(SUM(f.Cancelled) * 100.0 / COUNT(*), 1) AS cancel_pct
    FROM flights f
    JOIN airports ap ON f.Origin = ap.iata_code
    WHERE ap.iata_code IN ('ALB', 'LGA')
    GROUP BY ap.iata_code, ap.name
""").df()

In [ ]:
# Where do flights from ALB and LGA actually go?
con.sql("""
    SELECT
        f.Origin,
        f.Dest,
        ap.name AS dest_airport,
        ap.municipality AS dest_city,
        COUNT(*) AS flights
    FROM flights f
    JOIN airports ap ON f.Dest = ap.iata_code
    WHERE f.Origin IN ('ALB', 'LGA')
    GROUP BY f.Origin, f.Dest, ap.name, ap.municipality
    ORDER BY flights DESC
    LIMIT 10
""").df()

In [ ]:
# Who's flying out of ALB and LGA?
con.sql("""
    SELECT al.AIRLINE, COUNT(*) AS flights
    FROM flights f
    JOIN airlines al ON f.Reporting_Airline = al.IATA_CODE
    WHERE f.Origin IN ('ALB', 'LGA')
    GROUP BY al.AIRLINE
    ORDER BY flights DESC
""").df()

In [ ]:
# Monthly average departure delay, ALB vs LGA, 2022-2024
alb_lga = con.sql("""
    SELECT
        strftime(f.FlightDate::DATE, '%Y-%m') AS month,
        ap.iata_code                           AS airport,
        ROUND(AVG(f.DepDelay), 1)              AS avg_dep_delay
    FROM flights f
    JOIN airports ap ON f.Origin = ap.iata_code
    WHERE ap.iata_code IN ('ALB', 'LGA')
    GROUP BY month, airport
    ORDER BY month, airport
""").df()

pivot = alb_lga.pivot(index="month", columns="airport", values="avg_dep_delay")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(pivot.index, pivot["ALB"], marker="o", label="ALB", color="firebrick")
ax.plot(pivot.index, pivot["LGA"], marker="o", label="LGA", color="steelblue")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Average Departure Delay — ALB vs LGA (2022–2024)", fontsize=13)
ax.set_ylabel("Minutes")
ax.legend()
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.show()

## 8. Export to Parquet

Parquet is columnar — smaller files, faster queries.

In [ ]:
parquet_path = str(DATA / 'flights_all.parquet')

t0 = time.time()
con.sql(f"COPY flights TO '{parquet_path}' (FORMAT PARQUET)")
print(f'Written in {time.time()-t0:.1f}s')

csv_size  = sum(os.path.getsize(DATA / f'flights_{y}.csv') for y in [2022,2023,2024] if (DATA / f'flights_{y}.csv').exists())
parq_size = os.path.getsize(parquet_path)
print(f'CSV total:  {csv_size/1e6:.0f} MB')
print(f'Parquet:    {parq_size/1e6:.0f} MB  ({parq_size/csv_size*100:.0f}% of CSV size)')

In [ ]:
# Speed comparison — same query on CSV vs Parquet
query = """
    SELECT Reporting_Airline, COUNT(*), ROUND(AVG(ArrDelay),1)
    FROM '{source}'
    WHERE Cancelled = 0
    GROUP BY Reporting_Airline
"""

t0 = time.time()
con.sql(query.format(source=str(DATA / 'flights_*.csv'))).df()
csv_time = time.time() - t0

t0 = time.time()
con.sql(query.format(source=parquet_path)).df()
parq_time = time.time() - t0

print(f'CSV:     {csv_time:.2f}s')
print(f'Parquet: {parq_time:.2f}s  ({csv_time/parq_time:.1f}x faster)')

## 9. Bonus: Remote Files via httpfs

DuckDB can query files directly from S3 — no download, no server.

In [ ]:
# Same SQL, but the file lives on S3 — no download, no import
# Dataset: NYC Yellow Taxi 2019 — public bucket, no credentials needed
# Single partition file (~800K trips) so this runs in ~5s live instead of ~2min for the full 84M-row dataset
con.sql('INSTALL httpfs; LOAD httpfs;')

con.sql("""
    SELECT
        passenger_count,
        COUNT(*)                        AS trips,
        ROUND(AVG(trip_distance), 2)    AS avg_distance_mi,
        ROUND(AVG(total_amount), 2)     AS avg_fare_usd
    FROM read_parquet('s3://duckplyr-demo-taxi-data/taxi-data-2019-partitioned/month=6/data_0.parquet')
    WHERE passenger_count > 0
    GROUP BY passenger_count
    ORDER BY passenger_count
""").df()

---
## Wrap-up

Same tables, same SQL you just watched in the built-in UI — Python adds the DataFrame bridge and real charts on top, with zero extra infrastructure.

**Next up:** the Fabric notebooks (`fabric-workspace/`) — the same query patterns running against a Lakehouse in Microsoft Fabric, for when the data needs to live somewhere shared.